# 07 · ShellScope SQL layer (DuckDB Days 2–6)

Snapshot date: **2026-10-01**.

| Part | Day | Builds | Saved as |
|---|---|---|---|
| 0 | | Setup + helpers | |
| 1 | 2 | `clean_companies_v` **view** (rules) + `clean_companies` **table** (stored result) + cross-check vs pandas | `sql/01_clean_companies.sql` |
| 2 | 3 | `psc`, `psc_summary` tables + `company_psc` view (joins) | `sql/02_psc.sql` |
| 3 | 4 | `address_rank`, `mass_registration_addresses`, `address_window_features` (window functions) | `sql/03_address_ranking.sql` |
| 4 | 5 | `risk_flags` table (CTEs) + business questions Q1–Q6 | `sql/04_risk_flags.sql`, `sql/questions/`, `results/` |
| 5 | 6 | Speed (EXPLAIN ANALYZE, Parquet vs CSV), Q7–Q10, Parquet export, README section | `sql/questions/`, `docs/README_sql_section.md` |

⚠️ Shut down the kernels of other notebooks that use `shellscope.duckdb` first (file lock).
Run with **Kernel → Restart & Run All**. Every statement uses `CREATE OR REPLACE`, so rerunning is safe.

## 0. Setup

In [2]:
import duckdb, pandas as pd, zipfile, shutil
from pathlib import Path

ROOT    = Path("/Users/shraddha/Documents/DATA_SCIENCE/Shellscope/shellscope")
DB      = ROOT / "data" / "shellscope.duckdb"
SQL_DIR = ROOT / "sql"
PSC_DIR = ROOT / "data" / "raw" / "psc"
PARQ    = ROOT / "data" / "processed" / "shellscope_companies.parquet"
RESULTS = ROOT / "results"
Q_DIR   = SQL_DIR / "questions"
for d in (SQL_DIR, RESULTS, Q_DIR):
    d.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB))
con.execute("SET preserve_insertion_order = false")   # less memory for big CREATE TABLEs
con.execute("SET memory_limit = '6GB'")                # use '4GB' on an 8 GB Mac
pd.set_option("display.max_columns", 50)

def q(sql):
    """Run a query and return a pandas DataFrame."""
    return con.sql(sql).df()

def save_and_run(name, sql):
    """Save SQL to sql/<name> (the single source of truth), then run it."""
    path = SQL_DIR / name
    path.write_text(sql)
    con.execute(sql)
    print("ran and saved:", path.relative_to(ROOT))

print(q("SHOW TABLES"))

                name
0    clean_companies
1  clean_companies_v
2          companies
3                psc
4         psc_active
5      psc_portfolio


Column types decide how dates are cast. If the date columns are `VARCHAR` (text such as "31/12/2026"), the helper below switches to `try_strptime` automatically.

In [3]:
types = q("DESCRIBE companies").set_index("column_name")["column_type"]
print(types)

def as_date(col):
    """SQL expression turning column `col` into a DATE, whatever its stored type."""
    t = types[col].upper()
    if t.startswith("VARCHAR"):
        return f"""try_strptime("{col}", '%d/%m/%Y')::DATE"""
    return 'CAST("' + col + '" AS DATE)'

for c in ["IncorporationDate", "Accounts.NextDueDate", "ConfStmtNextDueDate"]:
    print(c, "->", as_date(c))

column_name
CompanyName                   VARCHAR
CompanyNumber                 VARCHAR
RegAddress.AddressLine1       VARCHAR
RegAddress.PostTown           VARCHAR
RegAddress.PostCode           VARCHAR
CompanyCategory               VARCHAR
CompanyStatus                 VARCHAR
IncorporationDate           TIMESTAMP
Accounts.NextDueDate        TIMESTAMP
Accounts.AccountCategory      VARCHAR
SICCode.SicText_1             VARCHAR
ConfStmtNextDueDate         TIMESTAMP
in_scope                      BOOLEAN
company_age_years              DOUBLE
accounts_overdue              BOOLEAN
confstmt_overdue              BOOLEAN
is_active                     BOOLEAN
strike_off_proposal           BOOLEAN
sic_code                      VARCHAR
sic_section                   VARCHAR
postcode_clean                VARCHAR
postcode_area                 VARCHAR
glasgow                       BOOLEAN
Name: column_type, dtype: str
IncorporationDate -> CAST("IncorporationDate" AS DATE)
Accounts.NextDueDate -> CAST(

---
## Part 1 · Cleaning in SQL (Day 2)

### 1a. Quick data-quality checks
**NULL** = unknown. `COUNT(col)` skips NULLs, so `COUNT(*) - COUNT(col)` = number of NULLs. `WHERE` keeps only TRUE rows, so NULL rows vanish from both `=` and `<>`.

In [3]:
q("""
SELECT COUNT(*)                                     AS rows,
       COUNT(*) - COUNT("RegAddress.AddressLine1")  AS null_line1,
       COUNT(*) - COUNT("RegAddress.PostTown")      AS null_town,
       COUNT(*) - COUNT("RegAddress.PostCode")      AS null_postcode,
       COUNT(*) - COUNT("Accounts.NextDueDate")     AS null_accounts_due,
       COUNT(*) - COUNT("SICCode.SicText_1")        AS null_sic
FROM companies
""")

,rows,null_line1,null_town,null_postcode,null_accounts_due,null_sic
0,5704711,63118,94569,91479,173523,0


In [4]:
# NULL logic, live
q("""
SELECT NULL = NULL                    AS eq,
       NULL IS NULL                   AS is_null,
       NULL IS DISTINCT FROM NULL     AS distinct_nulls,
       5 + NULL                       AS plus,
       'G' || NULL                    AS pipes,
       concat('G', NULL)              AS concat_,
       COALESCE(NULL, NULL, 'backup') AS coalesce_,
       NULLIF('', '')                 AS nullif_
""")

,eq,is_null,distinct_nulls,plus,pipes,concat_,coalesce_,nullif_
0,<NA>,True,False,<NA>,<NA>,G,backup,None


In [5]:
# The WHERE trap: eq + not_eq < total; the gap = NULL towns
q("""
SELECT (SELECT COUNT(*) FROM companies)                                            AS total,
       (SELECT COUNT(*) FROM companies WHERE "RegAddress.PostTown" =  'GLASGOW')   AS eq_glasgow,
       (SELECT COUNT(*) FROM companies WHERE "RegAddress.PostTown" <> 'GLASGOW')   AS not_glasgow,
       (SELECT COUNT(*) FROM companies WHERE "RegAddress.PostTown" IS NULL)        AS null_town
""")

,total,eq_glasgow,not_glasgow,null_town
0,5704711,75393,5534749,94569


In [6]:
# Postcodes: UK-shaped vs odd, and raw vs standardised distinct counts
UK_PC = '^[A-Z]{1,2}[0-9][A-Z0-9]? ?[0-9][A-Z]{2}$'
q(f"""
SELECT COUNT(*) FILTER (WHERE regexp_matches(upper(trim("RegAddress.PostCode")), '{UK_PC}')) AS uk_format,
       COUNT(*) FILTER (WHERE "RegAddress.PostCode" IS NOT NULL
                        AND NOT regexp_matches(upper(trim("RegAddress.PostCode")), '{UK_PC}')) AS odd_format,
       COUNT(DISTINCT "RegAddress.PostCode") AS distinct_raw,
       COUNT(DISTINCT NULLIF(regexp_replace(upper(trim("RegAddress.PostCode")), '[^A-Z0-9]', '', 'g'), '')) AS distinct_clean
FROM companies
""")

,uk_format,odd_format,distinct_raw,distinct_clean
0,5602414,10818,980212,979286


In [7]:
# TRY_CAST on SIC code (failed_cast ≈ 'None supplied'), untrimmed names, future dates
q(f"""
SELECT COUNT(*) - COUNT(TRY_CAST(substr(trim("SICCode.SicText_1"), 1, 5) AS INTEGER)) AS failed_sic_cast,
       COUNT(*) FILTER (WHERE CompanyName <> trim(CompanyName))                     AS names_extra_spaces,
       COUNT(*) FILTER (WHERE {as_date("IncorporationDate")} > DATE '2026-10-01')   AS future_inc_dates,
       MIN({as_date("IncorporationDate")}) AS earliest,
       MAX({as_date("IncorporationDate")}) AS latest
FROM companies
""")

,failed_sic_cast,names_extra_spaces,future_inc_dates,earliest,latest
0,231737,0,0,1327-01-01,2026-10-01


### 

1b. The cleaning rules (`clean_companies_v`, a view) + the stored result (`clean_companies`, a table)
A **view** is a saved query with no stored data: every query on it reruns the cleaning. That is slow on 5.7M rows, so the result is **materialised** once into the table `clean_companies`, and every later cell reads the table. Two **CTE** steps:
- `base` renames and casts the columns, and builds a clean postcode.
- `step2` derives the postcode area, SIC code and division, and the cleaned address line.

The final SELECT adds the address key, age, overdue flags, the Glasgow flag and the SIC section.

In [8]:
view_sql = f"""
-- clean_companies: one row per company with cleaned + derived columns.
-- Snapshot date 2026-10-01. Source table: companies.
CREATE OR REPLACE VIEW clean_companies_v AS
WITH base AS (
    SELECT
        CompanyNumber,
        trim(CompanyName)                         AS company_name,
        CompanyCategory,
        CompanyStatus,
        {as_date("IncorporationDate")}            AS incorporation_date,
        {as_date("Accounts.NextDueDate")}         AS accounts_next_due,
        {as_date("ConfStmtNextDueDate")}          AS confstmt_next_due,
        "RegAddress.AddressLine1"                 AS address_line1,
        upper(trim("RegAddress.PostTown"))        AS post_town,
        "SICCode.SicText_1"                       AS sic_text,
        NULLIF(regexp_replace(upper(trim("RegAddress.PostCode")), '[^A-Z0-9]', '', 'g'), '') AS postcode
    FROM companies
),
step2 AS (
    SELECT *,
        NULLIF(regexp_extract(postcode, '^[A-Z]{{1,2}}'), '')                AS postcode_area,
        NULLIF(regexp_extract(sic_text, '^\\s*([0-9]{{5}})', 1), '')         AS sic_code,
        TRY_CAST(substr(regexp_extract(sic_text, '^\\s*([0-9]{{5}})', 1), 1, 2) AS INTEGER) AS sic_division,
        regexp_replace(regexp_replace(
            trim(regexp_replace(regexp_replace(
                regexp_replace(upper(address_line1), '^C/O\\.?\\s*', ''),
            '[^A-Z0-9\\s/-]', ' ', 'g'), '\\s+', ' ', 'g')),
        ' ST$', ' STREET'), ' RD$', ' ROAD')                                AS line1_clean
    FROM base
)
SELECT
    *,
    line1_clean || ' | ' || postcode                                     AS address_key,
    date_diff('day', incorporation_date, DATE '2026-10-01') / 365.25     AS company_age_years,
    CASE WHEN accounts_next_due < DATE '2026-10-01' THEN 1 ELSE 0 END     AS accounts_overdue,
    CASE WHEN confstmt_next_due < DATE '2026-10-01' THEN 1 ELSE 0 END     AS confstmt_overdue,
    postcode_area = 'G'                                                  AS glasgow,
    CASE
        WHEN sic_text IS NULL OR upper(trim(sic_text)) LIKE 'NONE SUPPLIED%' THEN 'None supplied'
        WHEN sic_code = '99999'               THEN 'Dormant'
        WHEN sic_division BETWEEN 1  AND 3    THEN 'A'
        WHEN sic_division BETWEEN 5  AND 9    THEN 'B'
        WHEN sic_division BETWEEN 10 AND 33   THEN 'C'
        WHEN sic_division = 35                THEN 'D'
        WHEN sic_division BETWEEN 36 AND 39   THEN 'E'
        WHEN sic_division BETWEEN 41 AND 43   THEN 'F'
        WHEN sic_division BETWEEN 45 AND 47   THEN 'G'
        WHEN sic_division BETWEEN 49 AND 53   THEN 'H'
        WHEN sic_division BETWEEN 55 AND 56   THEN 'I'
        WHEN sic_division BETWEEN 58 AND 63   THEN 'J'
        WHEN sic_division BETWEEN 64 AND 66   THEN 'K'
        WHEN sic_division = 68                THEN 'L'
        WHEN sic_division BETWEEN 69 AND 75   THEN 'M'
        WHEN sic_division BETWEEN 77 AND 82   THEN 'N'
        WHEN sic_division = 84                THEN 'O'
        WHEN sic_division = 85                THEN 'P'
        WHEN sic_division BETWEEN 86 AND 88   THEN 'Q'
        WHEN sic_division BETWEEN 90 AND 93   THEN 'R'
        WHEN sic_division BETWEEN 94 AND 96   THEN 'S'
        WHEN sic_division BETWEEN 97 AND 98   THEN 'T'
        WHEN sic_division = 99                THEN 'U'
        ELSE 'Unknown'
    END                                                                  AS sic_section
FROM step2;

-- materialise: store the cleaned result once so later queries are fast
CREATE OR REPLACE TABLE clean_companies AS SELECT * FROM clean_companies_v;
"""
# an older version of this notebook made clean_companies a VIEW; that name now belongs to the table
if len(q("SELECT 1 FROM duckdb_views() WHERE view_name = 'clean_companies'")):
    con.execute("DROP VIEW clean_companies")
save_and_run("01_clean_companies.sql", view_sql)
q("""SELECT CompanyNumber, company_name, postcode, postcode_area, sic_code, sic_section,
            address_key, round(company_age_years, 2) AS age, accounts_overdue
     FROM clean_companies LIMIT 10""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ran and saved: sql/01_clean_companies.sql


,CompanyNumber,company_name,postcode,postcode_area,sic_code,sic_section,address_key,age,accounts_overdue
0,08209948,! LTD,HG11ND,HG,99999,Dormant,9 PRINCES SQUARE | HG11ND,14.05,0
1,16092999,!ABRIDGE TAX LTD,AL95BL,AL,62020,J,82 GREAT NORTH ROAD | AL95BL,1.86,0
2,11743365,!BIG IMPACT GRAPHICS LIMITED,EC1V2NX,EC,59112,J,124 CITY ROAD | EC1V2NX,7.76,0
3,16873705,!FE NETWORK LIMITED,CM201YS,CM,86210,Q,AACSL ACCOUNTANT LIMITED | CM201YS,0.85,0
4,15073164,!NFLECTION ADVISORY LIMITED,EN62DA,EN,70229,M,74 SANTERS LANE | EN62DA,3.13,0
5,13522064,!NFOGENIE LTD,WC2H9JQ,WC,58290,J,71-75 SHELTON STREET | WC2H9JQ,5.20,0
6,11006939,!NNOV8 LIMITED,TN85NF,TN,62090,J,OLD BARN FARM | TN85NF,8.97,0
7,SC606050,!NSPIRED INVESTMENTS LTD,AB117SY,AB,68209,L,26 POLMUIR ROAD | AB117SY,8.11,0
8,SC421617,!NSPIRED LTD,AB117SY,AB,70229,M,26 POLMUIR ROAD | AB117SY,14.47,1
9,FC031362,!OBAC LIMITED,NaN,NaN,NaN,None supplied,NaN,13.83,0


### 1c. Cross-check SQL vs pandas
First check that `CompanyNumber` is unique; otherwise the join would duplicate rows. Then count the rows where the SQL and pandas versions disagree. `IS DISTINCT FROM` is a NULL-safe "different". Most `diff_` values should be 0; any that aren't need explaining.

In [9]:
q("SELECT COUNT(*) AS rows, COUNT(DISTINCT CompanyNumber) AS distinct_numbers FROM companies")

,rows,distinct_numbers
0,5704711,5704711


In [10]:
q("""
SELECT COUNT(*) AS rows,
  SUM(CASE WHEN v.postcode         IS DISTINCT FROM c.postcode_clean                    THEN 1 ELSE 0 END) AS diff_postcode,
  SUM(CASE WHEN v.postcode_area    IS DISTINCT FROM c.postcode_area                     THEN 1 ELSE 0 END) AS diff_area,
  SUM(CASE WHEN v.glasgow          IS DISTINCT FROM c.glasgow                           THEN 1 ELSE 0 END) AS diff_glasgow,
  SUM(CASE WHEN v.accounts_overdue IS DISTINCT FROM CAST(c.accounts_overdue AS INTEGER)  THEN 1 ELSE 0 END) AS diff_overdue,
  SUM(CASE WHEN v.confstmt_overdue IS DISTINCT FROM CAST(c.confstmt_overdue AS INTEGER)  THEN 1 ELSE 0 END) AS diff_confstmt,
  SUM(CASE WHEN v.sic_section      IS DISTINCT FROM CAST(c.sic_section AS VARCHAR)      THEN 1 ELSE 0 END) AS diff_sic,
  MAX(abs(v.company_age_years - c.company_age_years))                                                AS max_age_gap
FROM clean_companies v
JOIN companies c USING (CompanyNumber)
""")

,rows,diff_postcode,diff_area,diff_glasgow,diff_overdue,diff_confstmt,diff_sic,max_age_gap
0,5704711,1514.0,1362.0,95498.0,0.0,0.0,16083.0,0.0


In [11]:
# Address key vs pandas Part 2 (2,632,476 distinct, ~1.6% missing)
q("""SELECT COUNT(DISTINCT address_key) AS distinct_addresses,
            round(100.0 * (COUNT(*) - COUNT(address_key)) / COUNT(*), 2) AS pct_missing_key
     FROM clean_companies""")

,distinct_addresses,pct_missing_key
0,2632365,1.61


In [12]:
# Inspect any sic_section mismatches (change the column to inspect others)
q("""
SELECT v.sic_text, v.sic_section AS sql_section, CAST(c.sic_section AS VARCHAR) AS pandas_section, COUNT(*) AS n
FROM clean_companies v JOIN companies c USING (CompanyNumber)
WHERE v.sic_section IS DISTINCT FROM CAST(c.sic_section AS VARCHAR)
GROUP BY ALL ORDER BY n DESC LIMIT 15
""")

,sic_text,sql_section,pandas_section,n
0,7499 - Non-trading company,Unknown,None supplied,1587
1,4521 - Gen construction & civil engineer,Unknown,None supplied,1531
2,7487 - Other business activities,Unknown,None supplied,1392
3,7011 - Development & sell real estate,Unknown,None supplied,699
4,9305 - Other service activities n.e.c.,Unknown,None supplied,514
5,4525 - Other special trades construction,Unknown,None supplied,423
6,2852 - General mechanical engineering,Unknown,None supplied,254
7,7415 - Holding Companies including Head Offices,Unknown,None supplied,253
8,7414 - Business & management consultancy,Unknown,None supplied,253
9,4531 - Installation electrical wiring etc.,Unknown,None supplied,249


---
## Part 2 · Joins + PSC ownership data (Day 3)

**PSC = Person with Significant Control** (more than 25% of shares or votes, or control by other means). The snapshot is **JSON Lines**, and one company can have many PSC rows.
- `LEFT JOIN` keeps all companies.
- `SEMI JOIN` keeps companies that have a match.
- `ANTI JOIN` keeps companies with no match.

⚠️ **Fan-out:** joining to raw PSC rows duplicates companies. Aggregate to one row per company first, then join.

### 2a. Check the PSC files
Put the Companies House PSC snapshot zip(s) into `data/raw/psc/` yourself. This cell **never moves anything**. It unzips PSC zips and keeps only files whose first line really is a PSC JSON record.

In [13]:
import json

PSC_DIR.mkdir(parents=True, exist_ok=True)

def looks_like_psc(path):
    """True if the file's first line is a PSC JSON record."""
    try:
        with open(path, encoding="utf-8") as fh:
            first = json.loads(fh.readline())
        return isinstance(first, dict) and "company_number" in first and "data" in first
    except Exception:
        return False

for z in PSC_DIR.rglob("*.zip"):                       # unzip only PSC zips
    name = z.name.lower()
    if "significant-control" in name or name.startswith("psc"):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(z.parent)
        print("unzipped:", z.name)

candidates = [p for p in PSC_DIR.rglob("*")
              if p.suffix.lower() in (".txt", ".json") and p.stat().st_size > 0]
psc_files = sorted(str(p) for p in candidates if looks_like_psc(p))
rejected  = [p.name for p in candidates if str(p) not in psc_files]

print(len(psc_files), "PSC files;", round(sum(Path(f).stat().st_size for f in psc_files) / 1e9, 2), "GB")
print("ignored (not PSC):", rejected)
assert psc_files, "No PSC files found: download the snapshot, put it in data/raw/psc, rerun."

unzipped: persons-with-significant-control-snapshot-2026-10-21.zip
1 PSC files; 14.04 GB
ignored (not PSC): []


### 2b. The `psc` table
`data->>'$.kind'` reads the `kind` field out of the nested JSON as text. Names are **never stored**: `md5` turns name + birth year/month into an ID. The SQL file keeps a `PSC_FILES` placeholder, so your Mac path never goes into git.

Set `REBUILD_PSC = True` only when you download a new snapshot; loading takes a few minutes.

In [14]:
REBUILD_PSC = "psc" not in set(q("SHOW TABLES")["name"])     # build only if missing

psc_sql = r"""
CREATE OR REPLACE TABLE psc AS
SELECT
    company_number                                   AS CompanyNumber,
    data->>'$.kind'                                  AS kind,
    TRY_CAST(data->>'$.notified_on' AS DATE)         AS notified_on,
    TRY_CAST(data->>'$.ceased_on'   AS DATE)         AS ceased_on,
    upper(data->>'$.country_of_residence')           AS country_of_residence,
    data->>'$.statement'                             AS statement,
    CAST(data->'$.natures_of_control' AS VARCHAR)    AS natures,
    -- privacy: hash of name + birth year/month as an identity proxy; names are not stored
    md5(lower(coalesce(data->>'$.name', '')) || '|' ||
        coalesce(data->>'$.date_of_birth.year', '') || '|' ||
        coalesce(data->>'$.date_of_birth.month', ''))  AS person_id
FROM read_json(PSC_FILES, format = 'newline_delimited',
               columns = {'company_number': 'VARCHAR', 'data': 'JSON'})
WHERE company_number IS NOT NULL;   -- drops the 'totals' summary line
"""
file_list = "[" + ", ".join(f"'{f}'" for f in psc_files) + "]"
if REBUILD_PSC:
    con.execute(psc_sql.replace("PSC_FILES", file_list))
    print("psc table built")
else:
    print("psc table already exists (set REBUILD_PSC = True to reload)")

q("""SELECT kind, COUNT(*) AS n, COUNT(*) FILTER (WHERE ceased_on IS NULL) AS still_active
     FROM psc GROUP BY kind ORDER BY n DESC""")

psc table already exists (set REBUILD_PSC = True to reload)


,kind,n,still_active
0,individual-person-with-significant-control,13987150,11918219
1,corporate-entity-person-with-significant-control,1081886,833807
2,persons-with-significant-control-statement,928899,567763
3,individual-beneficial-owner,34802,31409
4,legal-person-person-with-significant-control,17442,11063
5,corporate-entity-beneficial-owner,15817,13593
6,legal-person-beneficial-owner,591,524
7,super-secure-person-with-significant-control,380,283
8,super-secure-beneficial-owner,168,161
9,exemptions,107,107


### 2c. Match checks + the fan-out, in numbers
`with_psc_record + no_psc_record` should equal 5,704,711. `psc_orphans` are PSC rows for **dissolved** companies, which aren't in the live register. `fanout_rows` shows how a raw LEFT JOIN inflates the row count.

In [15]:
q("""
SELECT
 (SELECT COUNT(DISTINCT CompanyNumber) FROM psc)                                               AS psc_companies,
 (SELECT COUNT(*) FROM clean_companies c SEMI JOIN psc p USING (CompanyNumber))                AS with_psc_record,
 (SELECT COUNT(*) FROM clean_companies c ANTI JOIN psc p USING (CompanyNumber))                AS no_psc_record,
 (SELECT COUNT(DISTINCT CompanyNumber) FROM psc p ANTI JOIN companies c USING (CompanyNumber)) AS psc_orphans,
 (SELECT COUNT(*) FROM clean_companies c LEFT JOIN psc p USING (CompanyNumber))                AS fanout_rows
""")

,psc_companies,with_psc_record,no_psc_record,psc_orphans,fanout_rows
0,10987184,5552033,152678,5435151,9081056


In [16]:
print(q("""SELECT table_name, table_type FROM information_schema.tables
           WHERE table_name IN ('clean_companies','clean_companies_v','psc','psc_summary')"""))
print(q("SELECT COUNT(*) AS psc_rows FROM psc"))

          table_name  table_type
0    clean_companies  BASE TABLE
1                psc  BASE TABLE
2  clean_companies_v        VIEW
   psc_rows
0  16067242


### 2d. `psc_summary` (one row per company) + the `company_psc` view
- `portfolio` counts how many companies each person controls. It has one row per person, so joining it causes no fan-out.
- ⚠️ `NOT IN (...)` with a NULL country gives NULL, so that company is *not* counted as foreign.
- ⚠️ `person_id` is only a proxy: two people with the same name and birth month merge.

In [4]:
import time
con.execute("SET memory_limit = '6GB'")
con.execute("SET threads = 4")
con.execute("SET enable_progress_bar = true")

def run(label, sql):
    """Run one SQL statement and print how long it took."""
    t0 = time.perf_counter()
    con.execute(sql)
    print(f"{label}: {time.perf_counter() - t0:.1f} s")

In [5]:
run("A  psc_active", """
CREATE OR REPLACE TABLE psc_active AS
SELECT CompanyNumber, kind, statement, country_of_residence,
       CASE WHEN kind LIKE 'individual%' THEN person_id END AS ind_person_id,
       natures LIKE '%75-to-100%' AS owns_75
FROM psc
WHERE ceased_on IS NULL
""")
q("SELECT COUNT(*) AS active_rows, COUNT(ind_person_id) AS individual_rows FROM psc_active")

A  psc_active: 1.8 s


,active_rows,individual_rows
0,13376929,11949628


In [6]:
run("B  psc_portfolio", """
CREATE OR REPLACE TABLE psc_portfolio AS
SELECT ind_person_id AS person_id, COUNT(DISTINCT CompanyNumber) AS n_cos
FROM psc_active
WHERE ind_person_id IS NOT NULL
GROUP BY ind_person_id
""")
q("SELECT COUNT(*) AS people, MAX(n_cos) AS biggest_portfolio FROM psc_portfolio")

B  psc_portfolio: 0.7 s


,people,biggest_portfolio
0,8493856,1607


In [7]:
run("C1 psc_counts", """
CREATE OR REPLACE TABLE psc_counts AS
SELECT CompanyNumber,
  COUNT(*) FILTER (WHERE kind LIKE '%-person-with-significant-control')       AS n_active_psc,
  COUNT(*) FILTER (WHERE kind LIKE 'individual%')                             AS n_individual,
  COUNT(*) FILTER (WHERE kind LIKE 'corporate%')                              AS n_corporate,
  MAX(CASE WHEN statement LIKE 'no-individual-or-entity%' THEN 1 ELSE 0 END)  AS says_no_psc,
  COUNT(*) FILTER (WHERE kind LIKE 'individual%' AND country_of_residence NOT IN
        ('ENGLAND','SCOTLAND','WALES','NORTHERN IRELAND','UNITED KINGDOM','UK'))  AS n_foreign_resident,
  MAX(CASE WHEN owns_75 THEN 1 ELSE 0 END)                                    AS has_75pct_owner
FROM psc_active
GROUP BY CompanyNumber
""")
q("SELECT COUNT(*) AS companies FROM psc_counts")

C1 psc_counts: 1.0 s


,companies
0,10923560


In [8]:
plan = con.execute("""
EXPLAIN
SELECT a.CompanyNumber, MAX(pf.n_cos)
FROM psc_active a JOIN psc_portfolio pf ON a.ind_person_id = pf.person_id
GROUP BY a.CompanyNumber
""").fetchall()
print(plan[0][1])

┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│             #1            │
│                           │
│      ~5,622,692 rows      │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│    Aggregates: max(#1)    │
│                           │
│      ~5,622,692 rows      │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│       CompanyNumber       │
│           n_cos           │
│                           │
│      ~9,640,325 rows      │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_compress_string_│
│        uhugeint(#0)       │
│             #1            │
│                           │
│      ~9,640,325 rows      │
└─────────

In [9]:
run("C2 psc_company_portfolio", """
CREATE OR REPLACE TABLE psc_company_portfolio AS
SELECT a.CompanyNumber, MAX(pf.n_cos) AS max_psc_portfolio
FROM psc_active a
JOIN psc_portfolio pf ON a.ind_person_id = pf.person_id
GROUP BY a.CompanyNumber
""")

C2 psc_company_portfolio: 0.8 s


In [10]:
run("C3 psc_summary", """
CREATE OR REPLACE TABLE psc_summary AS
SELECT c.*, p.max_psc_portfolio
FROM psc_counts c
LEFT JOIN psc_company_portfolio p USING (CompanyNumber)
""")
q("SELECT COUNT(*) AS rows, COUNT(DISTINCT CompanyNumber) AS companies FROM psc_summary")

C3 psc_summary: 0.8 s


,rows,companies
0,10923560,10923560


In [11]:
run("D  company_psc view", """
CREATE OR REPLACE VIEW company_psc AS
SELECT c.*,
  p.CompanyNumber IS NOT NULL        AS has_psc_record,
  COALESCE(p.n_active_psc, 0)        AS n_active_psc,
  COALESCE(p.n_individual, 0)        AS n_individual,
  COALESCE(p.n_corporate, 0)         AS n_corporate,
  COALESCE(p.says_no_psc, 0)         AS says_no_psc,
  COALESCE(p.n_foreign_resident, 0)  AS n_foreign_resident,
  COALESCE(p.has_75pct_owner, 0)     AS has_75pct_owner,
  COALESCE(p.max_psc_portfolio, 0)   AS max_psc_portfolio
FROM clean_companies c
LEFT JOIN psc_summary p USING (CompanyNumber)
""")
t0 = time.perf_counter()
print(q("SELECT COUNT(*) AS rows FROM company_psc"), f"\ncheck: {time.perf_counter() - t0:.1f} s")

D  company_psc view: 0.0 s
      rows
0  5704711 
check: 0.2 s


In [12]:
old = (SQL_DIR / "02_psc.sql").read_text()
load_part = old.split("CREATE OR REPLACE TABLE psc_summary")[0]      # keep the psc load section
steps_sql = "\n".join(f"-- {lab}\n{sql};" for lab, sql in [
    ("A", "see notebook cell 2"), ("B", "see notebook cell 3")])      # placeholder removed below
(SQL_DIR / "02_psc.sql").write_text(load_part + """
-- A: active rows; person id for individuals only (keeps the later join a pure equality)
CREATE OR REPLACE TABLE psc_active AS SELECT CompanyNumber, kind, statement, country_of_residence,
  CASE WHEN kind LIKE 'individual%' THEN person_id END AS ind_person_id, natures LIKE '%75-to-100%' AS owns_75
FROM psc WHERE ceased_on IS NULL;
-- B: companies per individual
CREATE OR REPLACE TABLE psc_portfolio AS SELECT ind_person_id AS person_id, COUNT(DISTINCT CompanyNumber) AS n_cos
FROM psc_active WHERE ind_person_id IS NOT NULL GROUP BY ind_person_id;
-- C1-C3, D: see 07_sql_core.ipynb cells 4-8 (psc_counts, psc_company_portfolio, psc_summary, company_psc)
""")
print("saved: sql/02_psc.sql")

saved: sql/02_psc.sql


### 2d+. Practice: every join type on tiny tables (predict the row counts first!)
`l` has 5 companies (E5 has a NULL status). `r` has PSC counts: A1 appears **twice** (fan-out), Z9 isn't in `l`, and one key is NULL.
Predictions: INNER 3 · LEFT 6 · RIGHT 5 · FULL 8 · SEMI 2 · ANTI 3. NULL keys never match.

In [13]:
con.execute("""
CREATE OR REPLACE TEMP TABLE l AS
    SELECT * FROM (VALUES ('A1','Active'), ('B2','Strike'), ('C3','Active'), ('D4','Active'), ('E5', NULL)) t(id, status);
CREATE OR REPLACE TEMP TABLE r AS
    SELECT * FROM (VALUES ('A1', 2), ('A1', 3), ('C3', 1), ('Z9', 1), (NULL, 5)) t(id, n_psc);
""")
for join in ["INNER JOIN", "LEFT JOIN", "RIGHT JOIN", "FULL JOIN", "SEMI JOIN", "ANTI JOIN"]:
    n = len(q(f"SELECT * FROM l {join} r USING (id)"))
    print(f"{join:<11} -> {n} rows")

# the two classic ways to write an anti-join (both give B2, D4, E5)
print(q("SELECT l.id FROM l LEFT JOIN r ON l.id = r.id WHERE r.id IS NULL"))
print(q("SELECT id FROM l WHERE NOT EXISTS (SELECT 1 FROM r WHERE r.id = l.id)"))

INNER JOIN  -> 3 rows
LEFT JOIN   -> 6 rows
RIGHT JOIN  -> 5 rows
FULL JOIN   -> 8 rows
SEMI JOIN   -> 2 rows
ANTI JOIN   -> 3 rows
   id
0  B2
1  D4
2  E5
   id
0  B2
1  D4
2  E5


In [14]:
# How many companies have 0, 1, 2+ active PSCs?
q("""SELECT CASE WHEN n_active_psc = 0 THEN '0' WHEN n_active_psc = 1 THEN '1' ELSE '2+' END AS active_pscs,
            COUNT(*) AS companies,
            round(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
     FROM company_psc GROUP BY active_pscs ORDER BY active_pscs""")

,active_pscs,companies,pct
0,0,446481,7.83
1,1,3995533,70.04
2,2+,1262697,22.13


### 2e. Questions: ownership patterns

In [15]:
# Q1: no PSC record / 'no one controls us' statement, by status
q("""SELECT CompanyStatus, COUNT(*) AS n,
       round(100 * AVG(CASE WHEN NOT has_psc_record THEN 1 ELSE 0 END), 2) AS pct_no_record,
       round(100 * AVG(says_no_psc), 2)                                     AS pct_says_no_psc
     FROM company_psc GROUP BY CompanyStatus ORDER BY n DESC""")

,CompanyStatus,n,pct_no_record,pct_says_no_psc
0,Active,5166057,2.47,4.45
1,Active - Proposal to Strike off,423006,2.61,2.74
2,Liquidation,108330,25.91,2.29
3,In Administration,3709,4.93,4.88
4,Live but Receiver Manager on at least one charge,2156,38.78,0.97
5,Voluntary Arrangement,583,15.27,1.54
6,In Administration/Administrative Receiver,398,15.58,4.02
7,RECEIVERSHIP,182,96.70,0.00
8,ADMINISTRATION ORDER,121,100.00,0.00
9,ADMINISTRATIVE RECEIVER,104,96.15,0.00


In [16]:
# Q2: Glasgow vs rest of UK
q("""SELECT glasgow, COUNT(*) AS n,
       round(100 * AVG(CASE WHEN n_corporate > 0 THEN 1 ELSE 0 END), 2)        AS pct_corporate_owned,
       round(100 * AVG(CASE WHEN n_foreign_resident > 0 THEN 1 ELSE 0 END), 2) AS pct_foreign_psc
     FROM company_psc GROUP BY glasgow""")

,glasgow,n,pct_corporate_owned,pct_foreign_psc
0,True,83681,8.99,3.29
1,False,5525532,9.80,5.31
2,<NA>,95498,3.37,0.28


In [17]:
# Q3: does controlling many companies go with overdue accounts?
q("""SELECT CASE WHEN max_psc_portfolio = 0  THEN '0 (no individual PSC)'
                 WHEN max_psc_portfolio = 1  THEN '1'
                 WHEN max_psc_portfolio < 10 THEN '2-9'
                 WHEN max_psc_portfolio < 100 THEN '10-99'
                 ELSE '100+' END AS portfolio_band,
       COUNT(*) AS n, round(100 * AVG(accounts_overdue), 2) AS overdue_pct
     FROM company_psc GROUP BY portfolio_band ORDER BY portfolio_band""")

,portfolio_band,n,overdue_pct
0,0 (no individual PSC),917143,10.71
1,1,2385944,6.35
2,10-99,157942,10.29
3,100+,11176,5.91
4,2-9,2232506,7.80


---
## Part 3 · Window functions (Day 4)
`OVER (...)` calculates across related rows but **keeps every row**.
- `PARTITION BY` restarts the calculation for each group, and `ORDER BY` sets the order inside it.
- The frame (`ROWS` / `RANGE BETWEEN`) sets which rows are visible.
- `QUALIFY` filters on window results.

### 3a. `address_rank`, `mass_registration_addresses` (top 1%), `address_window_features`
- `same_addr_30d` counts the companies registered at the same address in the 30 days up to and including this one (a **burst** signal). It uses a numeric day number so the `RANGE` frame is simple.
- `days_since_prev` is the gap in days from the previous company at that address.

In [18]:
windows_sql = r"""
CREATE OR REPLACE TABLE address_rank AS
WITH a AS (
    SELECT address_key,
           any_value(postcode_area)                                         AS postcode_area,
           COUNT(*)                                                         AS n_companies,
           AVG(accounts_overdue)                                            AS overdue_rate,
           AVG(CASE WHEN CompanyStatus ILIKE '%strike%' THEN 1 ELSE 0 END)   AS strike_rate
    FROM clean_companies
    WHERE address_key IS NOT NULL
    GROUP BY address_key)
SELECT *,
    RANK()         OVER (ORDER BY n_companies DESC)                                   AS rank_n,
    PERCENT_RANK() OVER (ORDER BY n_companies DESC)                                   AS pct_rank,
    CAST(SUM(n_companies) OVER (ORDER BY n_companies DESC ROWS UNBOUNDED PRECEDING) AS DOUBLE)
      / CAST(SUM(n_companies) OVER () AS DOUBLE)                                       AS cum_share
FROM a;

-- the top 1% of addresses by number of companies
CREATE OR REPLACE TABLE mass_registration_addresses AS
SELECT * FROM address_rank WHERE pct_rank <= 0.01;

CREATE OR REPLACE TABLE address_window_features AS
WITH d AS (
    SELECT CompanyNumber, address_key, incorporation_date,
           date_diff('day', DATE '1300-01-01', incorporation_date) AS day_no
    FROM clean_companies
    WHERE address_key IS NOT NULL AND incorporation_date IS NOT NULL)
SELECT CompanyNumber, address_key, incorporation_date,
    ROW_NUMBER() OVER (PARTITION BY address_key ORDER BY incorporation_date, CompanyNumber)   AS arrival_no,
    COUNT(*)     OVER (PARTITION BY address_key ORDER BY day_no
                       RANGE BETWEEN 30 PRECEDING AND CURRENT ROW)                            AS same_addr_30d,
    day_no - LAG(day_no) OVER (PARTITION BY address_key ORDER BY day_no)                      AS days_since_prev
FROM d;
"""
save_and_run("03_address_ranking.sql", windows_sql)
q("""SELECT (SELECT COUNT(*) FROM address_rank) AS addresses,
            (SELECT MAX(n_companies) FROM address_rank) AS biggest,
            (SELECT COUNT(*) FROM mass_registration_addresses) AS mass_addresses,
            (SELECT MIN(n_companies) FROM mass_registration_addresses) AS min_cos_at_mass_address""")

ran and saved: sql/03_address_ranking.sql


,addresses,biggest,mass_addresses,min_cos_at_mass_address
0,2632365,87403,27176,13


### 3b. Checks against pandas + questions

In [19]:
# 1) pandas found ~351 addresses holding 11% of companies
q("SELECT MAX(rank_n) AS addresses_holding_11pct FROM address_rank WHERE cum_share <= 0.11")

,addresses_holding_11pct
0,353


In [20]:
# 2) Top 1% of addresses vs the rest (company-weighted rates)
q("""SELECT pct_rank <= 0.01 AS top_1pct, COUNT(*) AS addresses, SUM(n_companies) AS companies,
       round(100.0 * SUM(n_companies) / SUM(SUM(n_companies)) OVER (), 1) AS pct_of_all_companies,
       round(100 * SUM(overdue_rate * n_companies) / SUM(n_companies), 2)  AS overdue_pct,
       round(100 * SUM(strike_rate  * n_companies) / SUM(n_companies), 2)  AS strike_pct
     FROM address_rank GROUP BY top_1pct ORDER BY top_1pct DESC""")

,top_1pct,addresses,companies,pct_of_all_companies,overdue_pct,strike_pct
0,True,27176,2086180.0,37.2,7.83,5.35
1,False,2605189,3526962.0,62.8,7.80,8.82


In [21]:
# 3) Top 3 addresses per postcode area (notebook only, don't publish named lists)
q("""SELECT postcode_area, address_key, n_companies FROM address_rank
     WHERE postcode_area IS NOT NULL
     QUALIFY ROW_NUMBER() OVER (PARTITION BY postcode_area ORDER BY n_companies DESC) <= 3
     ORDER BY postcode_area, n_companies DESC""")

,postcode_area,address_key,n_companies
0,A,11 CORALS MEAD | A173AB,2
1,A,BERRYS LODGE | A82PF64,1
2,A,16 RODEN PLACE | A91N9CV,1
3,AA,19 BEVAN CLOSE RG30 4JD | AA99AA,1
4,AB,5 CARDEN PLACE | AB101UT,391
...,...,...,...
731,ZB,ATTRANS DEPOT | ZBG9018,1
732,ZE,MAIR S QUAY | ZE10PW,35
733,ZE,47 COMMERCIAL ROAD | ZE10NJ,13
734,ZE,2 NORTH NESS BUSINESS PARK | ZE10LZ,13


In [22]:
# 4) Monthly YoY % (LAG 12) + 3-month moving average; compare with pandas Part 4
q("""
WITH m AS (
    SELECT date_trunc('month', incorporation_date) AS month, COUNT(*) AS n
    FROM clean_companies
    WHERE incorporation_date >= DATE '2015-01-01' AND incorporation_date < DATE '2026-10-01'
    GROUP BY month)
SELECT month, n,
       round(100.0 * (n - LAG(n, 12) OVER (ORDER BY month)) / LAG(n, 12) OVER (ORDER BY month), 1) AS yoy_pct,
       round(AVG(n) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW))                AS ma3
FROM m ORDER BY month DESC LIMIT 15
""")

,month,n,yoy_pct,ma3
0,2026-09-01,68766,-4.3,65402.0
1,2026-08-01,58697,-7.5,63754.0
2,2026-07-01,68744,1.5,65017.0
3,2026-06-01,63821,2.4,64556.0
4,2026-05-01,62487,1.1,67772.0
5,2026-04-01,67360,6.4,66767.0
6,2026-03-01,73470,33.7,67623.0
7,2026-02-01,59472,30.1,59645.0
8,2026-01-01,69926,51.8,58851.0
9,2025-12-01,49537,47.7,59585.0


In [23]:
# 5) Registration bursts vs overdue accounts
q("""SELECT CASE WHEN f.same_addr_30d = 1  THEN '1 (alone)'
                 WHEN f.same_addr_30d < 10 THEN '2-9'
                 WHEN f.same_addr_30d < 100 THEN '10-99'
                 ELSE '100+' END AS burst_band,
       COUNT(*) AS n, round(100 * AVG(c.accounts_overdue), 2) AS overdue_pct
     FROM address_window_features f JOIN clean_companies c USING (CompanyNumber)
     GROUP BY burst_band ORDER BY burst_band""")

,burst_band,n,overdue_pct
0,1 (alone),4234644,8.23
1,10-99,211088,5.67
2,100+,242892,3.26
3,2-9,924518,7.55


### 3c. Practice: 5 window queries

In [24]:
# 1) Rank post towns by number of companies
q("""SELECT post_town, COUNT(*) AS n, RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
     FROM clean_companies WHERE post_town IS NOT NULL
     GROUP BY post_town ORDER BY rnk LIMIT 10""")

,post_town,n,rnk
0,LONDON,1111179,1
1,MANCHESTER,112616,2
2,BIRMINGHAM,99065,3
3,GLASGOW,75393,4
4,EDINBURGH,61238,5
5,BRISTOL,58631,6
6,LEEDS,54536,7
7,CARDIFF,53744,8
8,LIVERPOOL,49485,9
9,LEICESTER,47060,10


In [26]:
# 2) Top 3 SIC sections in each postcode area (ROW_NUMBER + QUALIFY)
q("""SELECT postcode_area, sic_section, COUNT(*) AS n
     FROM clean_companies WHERE postcode_area IS NOT NULL
     GROUP BY postcode_area, sic_section
     QUALIFY ROW_NUMBER() OVER (PARTITION BY postcode_area ORDER BY COUNT(*) DESC) <= 3
     ORDER BY postcode_area, n DESC
     LIMIT 30""")

,postcode_area,sic_section,n
0,A,None supplied,38
1,A,L,2
2,A,G,1
3,AA,F,1
4,AB,M,5483
5,AB,N,2565
6,AB,G,2555
7,AD,None supplied,3
8,AG,N,2
9,AG,G,2


In [27]:
# 3) Month-on-month change in incorporations (LAG 1)
q("""WITH m AS (SELECT date_trunc('month', incorporation_date) AS month, COUNT(*) AS n
               FROM clean_companies
               WHERE incorporation_date >= DATE '2025-01-01' AND incorporation_date < DATE '2026-10-01'
               GROUP BY month)
     SELECT month, n, n - LAG(n) OVER (ORDER BY month) AS change_vs_prev_month
     FROM m ORDER BY month""")

,month,n,change_vs_prev_month
0,2025-01-01,46067,<NA>
1,2025-02-01,45723,-344
2,2025-03-01,54966,9243
3,2025-04-01,63292,8326
4,2025-05-01,61793,-1499
5,2025-06-01,62351,558
6,2025-07-01,67718,5367
7,2025-08-01,63439,-4279
8,2025-09-01,71869,8430
9,2025-10-01,72130,261


In [28]:
# 4) Cumulative incorporations by year (running total of surviving companies)
q("""WITH y AS (SELECT year(incorporation_date) AS yr, COUNT(*) AS n
               FROM clean_companies WHERE incorporation_date >= DATE '2000-01-01'
               GROUP BY yr)
     SELECT yr, n, SUM(n) OVER (ORDER BY yr ROWS UNBOUNDED PRECEDING) AS cumulative
     FROM y ORDER BY yr""")

,yr,n,cumulative
0,2000,33113,33113.0
1,2001,34377,67490.0
2,2002,51578,119068.0
3,2003,66145,185213.0
4,2004,53303,238516.0
5,2005,56419,294935.0
6,2006,63655,358590.0
7,2007,69546,428136.0
8,2008,63789,491925.0
9,2009,70808,562733.0


In [29]:
# 5) NTILE(100): the top percentiles of addresses by company count
q("""WITH t AS (SELECT n_companies, NTILE(100) OVER (ORDER BY n_companies DESC) AS percentile FROM address_rank)
     SELECT percentile, COUNT(*) AS addresses, MIN(n_companies) AS min_cos, MAX(n_companies) AS max_cos
     FROM t GROUP BY percentile ORDER BY percentile LIMIT 5""")

,percentile,addresses,min_cos,max_cos
0,1,26324,13,87403
1,2,26324,6,13
2,3,26324,5,6
3,4,26324,4,5
4,5,26324,3,4


---
## Part 4 · Multi-step analysis with CTEs → `risk_flags` (Day 5)

- **CTE** (`WITH step1 AS (...), step2 AS (...) SELECT ...`): named steps that read top to bottom, like pandas method chaining.
- **Conditional aggregation:** `COUNT(*) FILTER (WHERE accounts_overdue = 1)`.
- **Percentiles:** `quantile_cont(x, 0.95)` is the value below which 95% of rows fall (it interpolates between values).
- **`GROUP BY ROLLUP (col)`** adds a **subtotal row** where `col` is NULL. `GROUPING(col) = 1` tells you which row is that subtotal (as opposed to a real NULL).

### 4a. Practice

In [30]:
# 1) Overdue rate by postcode area + a UK total row (ROLLUP)
q("""
WITH r AS (
    SELECT CASE WHEN GROUPING(postcode_area) = 1 THEN 'UK TOTAL'
                ELSE COALESCE(postcode_area, '(no postcode)') END            AS area,
           COUNT(*)                                                        AS n,
           COUNT(*) FILTER (WHERE accounts_overdue = 1)                    AS n_overdue,
           round(100.0 * COUNT(*) FILTER (WHERE accounts_overdue = 1) / COUNT(*), 2) AS overdue_pct
    FROM clean_companies
    GROUP BY ROLLUP (postcode_area))
SELECT * FROM r
WHERE n >= 1000
ORDER BY area = 'UK TOTAL' DESC, overdue_pct DESC
LIMIT 15
""")

,area,n,n_overdue,overdue_pct
0,UK TOTAL,5704711,440677,7.72
1,CF,82784,17919,21.65
2,NR,45084,5492,12.18
3,PR,40841,4922,12.05
4,OL,33747,3680,10.90
5,DD,15497,1650,10.65
6,LS,67126,7139,10.64
7,M,136822,14247,10.41
8,L,61312,6241,10.18
9,DN,43837,4437,10.12


In [31]:
# 2) Percentiles of companies per address
q("""SELECT quantile_cont(n_companies, 0.50) AS p50,
            quantile_cont(n_companies, 0.90) AS p90,
            quantile_cont(n_companies, 0.95) AS p95,
            quantile_cont(n_companies, 0.99) AS p99,
            MAX(n_companies)                AS max
     FROM address_rank""")

,p50,p90,p95,p99,max
0,1.0,2.0,3.0,13.0,87403


In [32]:
# 3) Share of companies with a corporate PSC, by SIC section
q("""SELECT sic_section, COUNT(*) AS n,
            round(100.0 * COUNT(*) FILTER (WHERE n_corporate > 0) / COUNT(*), 2) AS pct_corporate_psc
     FROM company_psc GROUP BY sic_section ORDER BY pct_corporate_psc DESC""")

,sic_section,n,pct_corporate_psc
0,D,17845,50.33
1,K,230692,24.04
2,Dormant,104507,23.88
3,B,8356,22.93
4,E,18313,14.19
5,C,235344,13.57
6,None supplied,231737,12.68
7,L,633620,12.05
8,N,394042,10.63
9,U,617,10.53


### 4b. The `risk_flags` table: one row per company, six 0/1 flags + `risk_score`

| Flag | 1 when |
|---|---|
| `at_mass_address` | the company's address is in `mass_registration_addresses` (top 1%) |
| `accounts_overdue` | accounts were due before 2026-10-01 |
| `confstmt_overdue` | the confirmation statement was due before 2026-10-01 |
| `corporate_psc` | at least one active PSC is a company |
| `psc_abroad` | at least one active individual PSC lives outside the UK |
| `no_psc_named` | no active PSC at all (no record, or only statements) |

`risk_score` = the sum of the six flags (0–6). It is a simple **count of warning signs**, not a probability. Every flag gets equal weight; that's a choice to state in the caveats.

In [33]:
risk_sql = r"""
-- risk_flags: one row per company, six 0/1 warning flags + risk_score (0-6).
CREATE OR REPLACE TABLE risk_flags AS
WITH mass AS (                         -- step 1: the top-1% addresses (one row each: no fan-out)
    SELECT address_key FROM mass_registration_addresses
),
flags AS (                             -- step 2: one row per company with the flags
    SELECT p.CompanyNumber, p.CompanyStatus, p.CompanyCategory,
           p.postcode_area, COALESCE(p.glasgow, false) AS glasgow,
           p.sic_section, p.address_key, p.incorporation_date, p.company_age_years,
           CASE WHEN m.address_key IS NOT NULL THEN 1 ELSE 0 END AS at_mass_address,
           p.accounts_overdue,
           p.confstmt_overdue,
           CASE WHEN p.n_corporate        > 0 THEN 1 ELSE 0 END AS corporate_psc,
           CASE WHEN p.n_foreign_resident > 0 THEN 1 ELSE 0 END AS psc_abroad,
           CASE WHEN p.n_active_psc       = 0 THEN 1 ELSE 0 END AS no_psc_named
    FROM company_psc p
    LEFT JOIN mass m ON p.address_key = m.address_key
)
SELECT *,                               -- step 3: add the score
       at_mass_address + accounts_overdue + confstmt_overdue
     + corporate_psc + psc_abroad + no_psc_named AS risk_score
FROM flags;
"""
save_and_run("04_risk_flags.sql", risk_sql)
q("SELECT COUNT(*) AS rows, COUNT(DISTINCT CompanyNumber) AS companies, round(AVG(risk_score), 3) AS avg_score FROM risk_flags")

ran and saved: sql/04_risk_flags.sql


,rows,companies,avg_score
0,5704711,5704711,0.795


### 4c. Business questions Q1–Q6
`answer()` saves each question's SQL to `sql/questions/qNN_*.sql` and its result to `results/qNN_*.csv`, and shows the table.

In [34]:
def answer(num, slug, question, sql):
    """Save the SQL (with the question as a comment) and the result CSV, then return the result."""
    sql_path = Q_DIR / f"q{num:02d}_{slug}.sql"
    csv_path = RESULTS / f"q{num:02d}_{slug}.csv"
    sql_path.write_text(f"-- Q{num}: {question}\n{sql.strip()}\n")
    df = q(sql)
    df.to_csv(csv_path, index=False)
    print(f"Q{num}: {question}\n  saved {sql_path.relative_to(ROOT)} and {csv_path.relative_to(ROOT)}")
    return df

In [35]:
answer(1, "mass_address_companies", "How many companies sit at the top 1% of addresses?", """
SELECT COUNT(*) FILTER (WHERE at_mass_address = 1)                              AS companies_at_mass_addresses,
       round(100.0 * COUNT(*) FILTER (WHERE at_mass_address = 1) / COUNT(*), 2) AS pct_of_all_companies,
       (SELECT COUNT(*) FROM mass_registration_addresses)                         AS mass_addresses
FROM risk_flags
""")

Q1: How many companies sit at the top 1% of addresses?
  saved sql/questions/q01_mass_address_companies.sql and results/q01_mass_address_companies.csv


,companies_at_mass_addresses,pct_of_all_companies,mass_addresses
0,2086180,36.57,27176


In [36]:
answer(2, "mass_vs_rest_overdue", "Are companies at mass addresses more often overdue?", """
SELECT at_mass_address, COUNT(*) AS n,
       round(100 * AVG(accounts_overdue), 2) AS accounts_overdue_pct,
       round(100 * AVG(confstmt_overdue), 2) AS confstmt_overdue_pct,
       round(median(company_age_years), 2)   AS median_age_years
FROM risk_flags GROUP BY at_mass_address ORDER BY at_mass_address DESC
""")

Q2: Are companies at mass addresses more often overdue?
  saved sql/questions/q02_mass_vs_rest_overdue.sql and results/q02_mass_vs_rest_overdue.csv


,at_mass_address,n,accounts_overdue_pct,confstmt_overdue_pct,median_age_years
0,1,2086180,7.83,11.87,6.33
1,0,3618531,7.67,12.82,5.69


In [37]:
answer(3, "postcode_area_risk", "Which postcode areas have the highest average risk score? (areas with 1,000+ companies)", """
SELECT postcode_area, COUNT(*) AS n,
       round(AVG(risk_score), 3)                                      AS avg_risk_score,
       round(100.0 * COUNT(*) FILTER (WHERE risk_score >= 3) / COUNT(*), 2) AS pct_score_3plus
FROM risk_flags
WHERE postcode_area IS NOT NULL
GROUP BY postcode_area HAVING COUNT(*) >= 1000
ORDER BY avg_risk_score DESC LIMIT 15
""")

Q3: Which postcode areas have the highest average risk score? (areas with 1,000+ companies)
  saved sql/questions/q03_postcode_area_risk.sql and results/q03_postcode_area_risk.csv


,postcode_area,n,avg_risk_score,pct_score_3plus
0,JE,7489,2.338,41.65
1,IM,3514,2.186,36.65
2,GY,3338,2.157,31.82
3,VG,3577,2.153,27.84
4,EC,200493,1.461,8.62
5,WC,144166,1.459,7.61
6,EH,80742,1.236,20.65
7,W,196421,1.197,5.64
8,HR,22032,1.027,3.64
9,N,165610,0.996,5.64


In [38]:
answer(4, "glasgow_vs_uk", "Is Glasgow above or below the UK average?", """
SELECT CASE WHEN GROUPING(glasgow) = 1 THEN 'UK total'
            WHEN glasgow THEN 'Glasgow (G)' ELSE 'Rest of UK' END AS area,
       COUNT(*) AS n,
       round(AVG(risk_score), 3)              AS avg_risk_score,
       round(100 * AVG(at_mass_address), 2)   AS mass_address_pct,
       round(100 * AVG(accounts_overdue), 2)  AS accounts_overdue_pct,
       round(100 * AVG(corporate_psc), 2)     AS corporate_psc_pct,
       round(100 * AVG(psc_abroad), 2)        AS psc_abroad_pct,
       round(100 * AVG(no_psc_named), 2)      AS no_psc_named_pct
FROM risk_flags
GROUP BY ROLLUP (glasgow)
ORDER BY 1      -- alphabetical: Glasgow (G), Rest of UK, UK total
""")

Q4: Is Glasgow above or below the UK average?
  saved sql/questions/q04_glasgow_vs_uk.sql and results/q04_glasgow_vs_uk.csv


,area,n,avg_risk_score,mass_address_pct,accounts_overdue_pct,corporate_psc_pct,psc_abroad_pct,no_psc_named_pct
0,Glasgow (G),83681,0.825,35.10,9.80,8.99,3.29,6.71
1,Rest of UK,5621030,0.794,36.59,7.69,9.69,5.23,7.84
2,UK total,5704711,0.795,36.57,7.72,9.68,5.20,7.83


In [39]:
answer(5, "risk_score_distribution", "How many companies combine several warning signs?", """
SELECT risk_score, COUNT(*) AS companies,
       round(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS pct,
       round(100.0 * SUM(COUNT(*)) OVER (ORDER BY risk_score DESC ROWS UNBOUNDED PRECEDING)
             / SUM(COUNT(*)) OVER (), 3)                   AS pct_this_score_or_higher
FROM risk_flags GROUP BY risk_score ORDER BY risk_score
""")

Q5: How many companies combine several warning signs?
  saved sql/questions/q05_risk_score_distribution.sql and results/q05_risk_score_distribution.csv


,risk_score,companies,pct,pct_this_score_or_higher
0,0,2583470,45.287,100.000
1,1,1998261,35.028,54.713
2,2,862663,15.122,19.685
3,3,231427,4.057,4.563
4,4,28776,0.504,0.506
5,5,114,0.002,0.002


In [40]:
answer(6, "sic_high_risk", "Which industries (SIC sections) have the largest share of companies with 3+ warning signs?", """
SELECT sic_section, COUNT(*) AS n,
       COUNT(*) FILTER (WHERE risk_score >= 3)                              AS n_score_3plus,
       round(100.0 * COUNT(*) FILTER (WHERE risk_score >= 3) / COUNT(*), 2) AS pct_score_3plus
FROM risk_flags GROUP BY sic_section ORDER BY pct_score_3plus DESC
""")

Q6: Which industries (SIC sections) have the largest share of companies with 3+ warning signs?
  saved sql/questions/q06_sic_high_risk.sql and results/q06_sic_high_risk.csv


,sic_section,n,n_score_3plus,pct_score_3plus
0,Unknown,16083,14340,89.16
1,None supplied,231737,50201,21.66
2,U,617,51,8.27
3,B,8356,681,8.15
4,D,17845,1381,7.74
5,I,299004,15791,5.28
6,N,394042,18754,4.76
7,Dormant,104507,4915,4.70
8,E,18313,859,4.69
9,C,235344,10556,4.49


---
## Part 5 · Speed + deliverable (Day 6)

### 5a. How DuckDB runs a query: `EXPLAIN ANALYZE`
`EXPLAIN` shows the **plan**, the steps DuckDB will take. `EXPLAIN ANALYZE` **runs** the query and adds the real time and row count of each step. Read it **bottom-up**: scans (reading data) at the bottom, then filters, joins and aggregates, with the result at the top. Look for the step with the most time, and for joins producing more rows than expected (fan-out!).

In [41]:
plan = con.execute("""
EXPLAIN ANALYZE
SELECT c.postcode_area, AVG(r.risk_score) AS avg_score
FROM clean_companies c JOIN risk_flags r USING (CompanyNumber)
GROUP BY c.postcode_area
""").fetchall()
print(plan[-1][-1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 EXPLAIN ANALYZE SELECT c.postcode_area, AVG(r.risk_score) AS avg_score FROM clean_companies c JOIN risk_flags r USING (CompanyNumber) GROUP BY c.postcode_area 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.127s              ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ───────────────────

### 5b. Parquet vs CSV: time the same query on each
**Parquet** stores data **by column**, compressed, with types saved. A query reading 1 column reads only that column. **CSV** is plain text by row: the whole file must be read and parsed every time. The test writes a temporary CSV copy (about 1–2 GB), times the same query on both, then deletes it.

In [42]:
import time

CSV_TMP = ROOT / "data" / "processed" / "companies_tmp.csv"
con.execute(f"COPY (SELECT * FROM read_parquet('{PARQ}')) TO '{CSV_TMP}' (HEADER, DELIMITER ',')")

def timed(label, sql, repeats=3):
    """Best of `repeats` runs, in seconds."""
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        con.sql(sql).fetchall()
        best = min(best, time.perf_counter() - t0)
    print(f"{label:<8} {best:6.2f} s")
    return best

query = "SELECT CompanyStatus, COUNT(*) FROM {src} GROUP BY CompanyStatus"
t_parq = timed("Parquet", query.format(src=f"read_parquet('{PARQ}')"))
t_csv  = timed("CSV",     query.format(src=f"read_csv_auto('{CSV_TMP}')"))
print(f"CSV is {t_csv / t_parq:.1f}x slower;",
      f"file sizes: Parquet {PARQ.stat().st_size/1e6:.0f} MB vs CSV {CSV_TMP.stat().st_size/1e6:.0f} MB")

CSV_TMP.unlink()          # delete the temporary CSV

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Parquet    0.01 s
CSV        0.78 s
CSV is 79.5x slower; file sizes: Parquet 338 MB vs CSV 1708 MB


### 5c. Interview-style practice on our own data
These are the three classic patterns StrataScratch asks about: **top-N per group**, **running total**, **self-join**.

In [43]:
# Top-N per group: the 2 oldest active companies in each SIC section
q("""SELECT sic_section, CompanyNumber, incorporation_date
     FROM clean_companies
     WHERE CompanyStatus = 'Active' AND incorporation_date IS NOT NULL
     QUALIFY ROW_NUMBER() OVER (PARTITION BY sic_section ORDER BY incorporation_date) <= 2
     ORDER BY sic_section, incorporation_date""")

,sic_section,CompanyNumber,incorporation_date
0,A,00011066,1876-12-15
1,A,00011771,1877-10-20
2,B,00009433,1875-04-23
3,B,SC000792,1877-11-06
4,C,00000140,1856-09-30
5,C,00001812,1865-01-11
6,D,00151665,1918-10-14
7,D,00174142,1921-04-09
8,Dormant,00001350,1864-06-14
9,Dormant,00001364,1864-06-20


In [44]:
# Running total: Glasgow incorporations per month in 2026 and the year-to-date total
q("""WITH m AS (SELECT date_trunc('month', incorporation_date) AS month, COUNT(*) AS n
               FROM clean_companies
               WHERE glasgow AND incorporation_date >= DATE '2026-01-01' AND incorporation_date < DATE '2026-10-01'
               GROUP BY month)
     SELECT month, n, SUM(n) OVER (ORDER BY month ROWS UNBOUNDED PRECEDING) AS year_to_date
     FROM m ORDER BY month""")

,month,n,year_to_date
0,2026-01-01,1082,1082.0
1,2026-02-01,1036,2118.0
2,2026-03-01,1155,3273.0
3,2026-04-01,1057,4330.0
4,2026-05-01,947,5277.0
5,2026-06-01,1001,6278.0
6,2026-07-01,955,7233.0
7,2026-08-01,994,8227.0
8,2026-09-01,1110,9337.0


In [45]:
# Self-join: each month vs the same month last year, by joining the monthly table to itself (no LAG)
q("""WITH m AS (SELECT date_trunc('month', incorporation_date) AS month, COUNT(*) AS n
               FROM clean_companies
               WHERE incorporation_date >= DATE '2024-01-01' AND incorporation_date < DATE '2026-10-01'
               GROUP BY month)
     SELECT cur.month, cur.n, prev.n AS n_last_year,
            round(100.0 * (cur.n - prev.n) / prev.n, 1) AS yoy_pct
     FROM m AS cur
     JOIN m AS prev ON prev.month = cur.month - INTERVAL 1 YEAR
     ORDER BY cur.month DESC LIMIT 12""")

,month,n,n_last_year,yoy_pct
0,2026-09-01,68766,71869,-4.3
1,2026-08-01,58697,63439,-7.5
2,2026-07-01,68744,67718,1.5
3,2026-06-01,63821,62351,2.4
4,2026-05-01,62487,61793,1.1
5,2026-04-01,67360,63292,6.4
6,2026-03-01,73470,54966,33.7
7,2026-02-01,59472,45723,30.1
8,2026-01-01,69926,46067,51.8
9,2025-12-01,49537,33528,47.7


### 5d. Business questions Q7–Q10

In [46]:
answer(7, "sic_no_owner", "Which SIC sections have the most companies with no named owner?", """
SELECT sic_section, COUNT(*) AS n,
       COUNT(*) FILTER (WHERE no_psc_named = 1)                              AS n_no_psc_named,
       round(100.0 * COUNT(*) FILTER (WHERE no_psc_named = 1) / COUNT(*), 2) AS pct_no_psc_named
FROM risk_flags GROUP BY sic_section ORDER BY n_no_psc_named DESC
""")

Q7: Which SIC sections have the most companies with no named owner?
  saved sql/questions/q07_sic_no_owner.sql and results/q07_sic_no_owner.csv


,sic_section,n,n_no_psc_named,pct_no_psc_named
0,None supplied,231737,172169,74.29
1,T,128100,70874,55.33
2,L,633620,28551,4.51
3,M,676018,17342,2.57
4,Unknown,16083,16065,99.89
5,S,234864,15657,6.67
6,N,394042,14303,3.63
7,R,131211,13553,10.33
8,G,743696,13012,1.75
9,J,475327,12995,2.73


In [47]:
answer(8, "corporate_owner_by_year", "How has the share of corporate owners changed by incorporation year?", """
SELECT year(incorporation_date) AS inc_year, COUNT(*) AS n,
       round(100 * AVG(corporate_psc), 2) AS corporate_psc_pct
FROM risk_flags
WHERE incorporation_date >= DATE '2000-01-01' AND incorporation_date < DATE '2026-10-01'
GROUP BY inc_year ORDER BY inc_year
""")

Q8: How has the share of corporate owners changed by incorporation year?
  saved sql/questions/q08_corporate_owner_by_year.sql and results/q08_corporate_owner_by_year.csv


,inc_year,n,corporate_psc_pct
0,2000,33113,21.56
1,2001,34377,20.11
2,2002,51578,15.70
3,2003,66145,13.78
4,2004,53303,15.78
5,2005,56419,15.53
6,2006,63655,15.56
7,2007,69546,14.80
8,2008,63789,14.19
9,2009,70808,13.74


In [48]:
# PRIVATE: named addresses. results/q09_* is added to .gitignore; don't publish it.
answer(9, "top_risk_addresses", "Which 20 addresses (20+ companies) have the highest average risk score?", """
SELECT address_key, any_value(postcode_area) AS postcode_area, COUNT(*) AS n,
       round(AVG(risk_score), 2)             AS avg_risk_score,
       round(100 * AVG(accounts_overdue), 1) AS overdue_pct,
       round(100 * AVG(no_psc_named), 1)     AS no_psc_named_pct
FROM risk_flags
WHERE address_key IS NOT NULL
GROUP BY address_key HAVING COUNT(*) >= 20
ORDER BY avg_risk_score DESC, n DESC LIMIT 20
""")

Q9: Which 20 addresses (20+ companies) have the highest average risk score?
  saved sql/questions/q09_top_risk_addresses.sql and results/q09_top_risk_addresses.csv


,address_key,postcode_area,n,avg_risk_score,overdue_pct,no_psc_named_pct
0,OFFICE 3 | M456AT,M,49,4.0,100.0,0.0
1,OFFICE H ENERGY HOUSE | WS136DP,WS,42,4.0,100.0,0.0
2,OFFICE 6 MCF COMPLEX 60 | DY101AQ,DY,39,4.0,100.0,0.0
3,UNIT 4E CENTRAL PARK HALESOWEN ROAD | DY29NW,DY,38,4.0,100.0,0.0
4,UNIT 3 | NG316LU,NG,38,4.0,100.0,0.0
5,PWC 8TH FLOOR CENTRAL SQUARE | LS14DL,LS,33,4.0,100.0,0.0
6,FIRST FLOOR OFFICES | B646PL,B,32,4.0,100.0,0.0
7,BDO STOY HAYWARD LLP | W1U7EU,W,30,4.0,100.0,100.0
8,RS2-6 IVY BUSINESS CENTRE | M359BG,M,30,4.0,100.0,0.0
9,OFFICE 3 AND 4 MINISTER HOUSE 88-89 | WV14EX,WV,29,4.0,100.0,0.0


In [49]:
answer(10, "flag_combinations", "Which flags occur together most often? (companies with 2+ flags)", """
SELECT concat_ws(' + ',
           CASE WHEN at_mass_address  = 1 THEN 'mass address' END,
           CASE WHEN accounts_overdue = 1 THEN 'accounts overdue' END,
           CASE WHEN confstmt_overdue = 1 THEN 'conf. stmt overdue' END,
           CASE WHEN corporate_psc    = 1 THEN 'corporate PSC' END,
           CASE WHEN psc_abroad       = 1 THEN 'PSC abroad' END,
           CASE WHEN no_psc_named     = 1 THEN 'no PSC named' END) AS flag_combination,
       COUNT(*) AS companies,
       round(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_flagged
FROM risk_flags WHERE risk_score >= 2
GROUP BY flag_combination ORDER BY companies DESC LIMIT 10
""")

Q10: Which flags occur together most often? (companies with 2+ flags)
  saved sql/questions/q10_flag_combinations.sql and results/q10_flag_combinations.csv


,flag_combination,companies,pct_of_flagged
0,mass address + corporate PSC,260138,23.16
1,accounts overdue + conf. stmt overdue,164300,14.63
2,mass address + PSC abroad,158520,14.12
3,mass address + no PSC named,104022,9.26
4,mass address + accounts overdue + conf. stmt o...,89200,7.94
5,mass address + conf. stmt overdue,70040,6.24
6,accounts overdue + conf. stmt overdue + no PSC...,39869,3.55
7,conf. stmt overdue + no PSC named,37892,3.37
8,mass address + conf. stmt overdue + PSC abroad,26141,2.33
9,mass address + conf. stmt overdue + no PSC named,26103,2.32


### 5e. Export `risk_flags` + the README section
`COPY ... TO ... (FORMAT parquet)` writes a table to a Parquet file (gitignored; Network Analysis week reads it). The README helper builds a Markdown table of the 10 questions with links to each `.sql` file. Fill in the one-line answers, then paste it into `README.md`.

In [50]:
RF_PARQ = ROOT / "data" / "processed" / "risk_flags.parquet"
con.execute(f"COPY risk_flags TO '{RF_PARQ}' (FORMAT parquet)")
print("exported:", RF_PARQ.relative_to(ROOT), f"{RF_PARQ.stat().st_size/1e6:.0f} MB")

rows = ["## SQL analysis (DuckDB)", "",
        "Cleaned in SQL (`sql/01_clean_companies.sql`), joined to PSC ownership data (`sql/02_psc.sql`), "
        "addresses ranked with window functions (`sql/03_address_ranking.sql`), and six warning flags combined "
        "into `risk_flags` (`sql/04_risk_flags.sql`). Snapshot 2026-10-01; risk indicators, not proof.", "",
        "| # | Question | Answer | SQL |", "|---|---|---|---|"]
for p in sorted(Q_DIR.glob("q*.sql")):
    question = p.read_text().splitlines()[0].split(":", 1)[1].strip()
    rows.append(f"| {p.name[1:3]} | {question} | _fill in, with a number_ | [{p.name}](sql/questions/{p.name}) |")
(ROOT / "docs").mkdir(exist_ok=True)
(ROOT / "docs" / "README_sql_section.md").write_text("\n".join(rows) + "\n")
print("\n".join(rows))

exported: data/processed/risk_flags.parquet 193 MB
## SQL analysis (DuckDB)

Cleaned in SQL (`sql/01_clean_companies.sql`), joined to PSC ownership data (`sql/02_psc.sql`), addresses ranked with window functions (`sql/03_address_ranking.sql`), and six warning flags combined into `risk_flags` (`sql/04_risk_flags.sql`). Snapshot 2026-10-01; risk indicators, not proof.

| # | Question | Answer | SQL |
|---|---|---|---|
| 01 | How many companies sit at the top 1% of addresses? | _fill in, with a number_ | [q01_mass_address_companies.sql](sql/questions/q01_mass_address_companies.sql) |
| 02 | Are companies at mass addresses more often overdue? | _fill in, with a number_ | [q02_mass_vs_rest_overdue.sql](sql/questions/q02_mass_vs_rest_overdue.sql) |
| 03 | Which postcode areas have the highest average risk score? (areas with 1,000+ companies) | _fill in, with a number_ | [q03_postcode_area_risk.sql](sql/questions/q03_postcode_area_risk.sql) |
| 04 | Is Glasgow above or below the UK average? |

---
## Findings (fill in after running)
- **Cleaning:** SQL vs pandas differences: …
- **Ownership:** …% of companies name no PSC; …% have a corporate PSC; Glasgow vs UK: …
- **Addresses:** top 1% of addresses hold …% of companies; overdue …% vs …%.
- **Risk score:** …% of companies have 3+ flags; the most common combination is …

## Caveats
- Live register only: dissolved companies are missing (survivorship bias); PSC orphans confirm this.
- `person_id` (name + birth month) is a proxy and can merge different people.
- `risk_score` gives every flag equal weight, and some flags have innocent explanations (e.g. listed companies can be exempt from naming a PSC; a corporate owner is often a normal parent company).
- These are **risk indicators, not proof** of wrongdoing. Don't publish named address lists.

In [51]:
print(q("SHOW TABLES"))
con.close()   # release the file lock

                           name
0                  address_rank
1       address_window_features
2               clean_companies
3             clean_companies_v
4                     companies
5                   company_psc
6                             l
7   mass_registration_addresses
8                           psc
9                    psc_active
10        psc_company_portfolio
11                   psc_counts
12                psc_portfolio
13                  psc_summary
14                            r
15                   risk_flags


**Terminal: commit**
```bash
cd /Users/shraddha/Documents/DATA_SCIENCE/Shellscope/shellscope
grep -qx "data/raw/" .gitignore || echo "data/raw/" >> .gitignore
grep -qx "data/processed/" .gitignore || echo "data/processed/" >> .gitignore
grep -qx "results/q09_top_risk_addresses.csv" .gitignore || echo "results/q09_top_risk_addresses.csv" >> .gitignore
git rm --cached -q sql/03_windows.sql 2>/dev/null; rm -f sql/03_windows.sql
git add .gitignore sql/ results/ docs/ notebooks/07_sql_core.ipynb README.md
git status
git commit -m "SQL layer: cleaning view, PSC joins, address ranking, risk_flags + 10 business questions"
git push
```